# MCSL-069 — Question 13

**Section:** 8.1 AI, Session 8, Exercise a

**Question:** Classify a propositional formula as valid, satisfiable, or unsatisfiable.

The accompanying CSV file is a small, reproducible input dataset. Run the code cell below to see the implementation and its results.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import ast
formula = pd.read_csv("a_formula.csv").loc[0, "formula"]
tree = ast.parse(formula, mode="eval").body
variables = sorted({node.id for node in ast.walk(tree) if isinstance(node, ast.Name)})
def evaluate(node, assignment):
    if isinstance(node, ast.Name):
        return assignment[node.id]
    if isinstance(node, ast.Constant) and isinstance(node.value, bool):
        return node.value
    if isinstance(node, ast.BoolOp):
        values = [evaluate(value, assignment) for value in node.values]
        return all(values) if isinstance(node.op, ast.And) else any(values)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.Not):
        return not evaluate(node.operand, assignment)
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

results = []
for bits in range(2 ** len(variables)):
    assignment = {name: bool(bits & (1 << i)) for i, name in enumerate(variables)}
    results.append((assignment, evaluate(tree, assignment)))
valid = all(value for _, value in results)
satisfiable = any(value for _, value in results)
print("Formula:", formula)
print("Truth table:", results)
print("Valid:", valid, "| Satisfiable:", satisfiable,
      "| Classification:", "valid" if valid else ("satisfiable" if satisfiable else "unsatisfiable"))


Formula: (P or Q) and (not P or not Q)
Truth table: [({'P': False, 'Q': False}, False), ({'P': True, 'Q': False}, True), ({'P': False, 'Q': True}, True), ({'P': True, 'Q': True}, False)]
Valid: False | Satisfiable: True | Classification: satisfiable
